# 第 4 章 · 传感器：知道自己在哪，看见外面的世界

配套讲义：仓库里 `robot-system-architecture/notes/lessons/04-sensors.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（互补滤波的信号和相机参数是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第 3 步 延迟误差 | $e = v\Delta t$ | `delay_error(0.5, 0.060)` | 3.0 cm |
| 第 3 步 补偿 | $\hat x = x_{obs} + v\Delta t$ | `compensate` | 0.200 m → 0.230 m |
| 第 4 步 只用陀螺 | 积分 $\omega$ | `np.cumsum(gyro) * dt` | 10 秒漂 0.200 rad |
| 第 4 步 只用加速度计 | $\theta^{acc}$ | `acc` | 均方根误差 0.050 rad |
| 第 4 步 互补滤波 | $\alpha(\hat\theta+\omega\Delta t) + (1-\alpha)\theta^{acc}$ | `complementary(..., alpha=0.95)` | 均方根误差 0.008 rad |
| 公式表 针孔投影 | $u = f_xX/Z + c_x$ | `project` | (0.1, −0.05, 1) m → (380, 210) 像素 |

互补滤波的信号（真实角度、陀螺零偏、加速度计噪声）和相机参数都是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `delay_error`**：延迟误差 e = v · Δt。

In [ ]:
def delay_error(v: float, dt: float) -> float:
    """延迟误差 e = v · Δt。"""
    return v * dt

**函数 `compensate`**：延迟补偿 x̂ = x_obs + v · Δt。

In [ ]:
def compensate(x_obs: float, v: float, dt: float) -> float:
    """延迟补偿 x̂ = x_obs + v · Δt。"""
    return x_obs + v * dt

**函数 `project`**：针孔相机投影 u = f_x X / Z + c_x，v = f_y Y / Z + c_y。

In [ ]:
def project(point: np.ndarray, fx: float, fy: float, cx: float, cy: float) -> tuple[float, float]:
    """针孔相机投影 u = f_x X / Z + c_x，v = f_y Y / Z + c_y。"""
    X, Y, Z = point
    return fx * X / Z + cx, fy * Y / Z + cy

**函数 `complementary`**：互补滤波 θ̂_k = α (θ̂_{k−1} + ω_k Δt) + (1 − α) θ_acc,k。

In [ ]:
def complementary(gyro: np.ndarray, acc_angle: np.ndarray, dt: float, alpha: float) -> np.ndarray:
    """互补滤波 θ̂_k = α (θ̂_{k−1} + ω_k Δt) + (1 − α) θ_acc,k。"""
    est = np.zeros_like(gyro)
    est[0] = acc_angle[0]
    for k in range(1, len(gyro)):
        est[k] = alpha * (est[k - 1] + gyro[k] * dt) + (1 - alpha) * acc_angle[k]
    return est

**函数 `fake_imu`**：编一段 10 秒的一维姿态：真实角度慢慢摆，陀螺有零偏 0.02 rad/s，加速度计噪声 0.05 rad。

In [ ]:
def fake_imu(n: int = 2000, dt: float = 0.005, seed: int = 0) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """编一段 10 秒的一维姿态：真实角度慢慢摆，陀螺有零偏 0.02 rad/s，加速度计噪声 0.05 rad。"""
    rng = np.random.default_rng(seed)
    t = np.arange(n) * dt
    truth = 0.3 * np.sin(0.5 * t)
    gyro = np.gradient(truth, dt) + 0.02  # 角速度 + 零偏
    acc = truth + rng.normal(0, 0.05, n)  # 角度 + 白噪声
    return truth, gyro, acc

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第三步：时间同步——延迟误差 e = v Δt")
e = delay_error(0.5, 0.060)
print(f"  传送带 0.5 m/s、延迟 60 ms → 差 {e * 100:.1f} cm")
print(f"  按时间戳补偿：图里看到 x = 0.200 m → 预测 x̂ = {compensate(0.200, 0.5, 0.060):.3f} m")
print(f"  延迟拖到 0：差 {delay_error(0.5, 0.0) * 100:.1f} cm")

In [ ]:
print("\n第四步：互补滤波——只用陀螺、只用加速度计、两者融合")
dt = 0.005
truth, gyro, acc = fake_imu(dt=dt)
gyro_only = truth[0] + np.cumsum(gyro) * dt
fused = complementary(gyro, acc, dt, alpha=0.95)
def rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean((x - truth) ** 2)))
print(f"  只积分陀螺：10 秒后误差 {abs(gyro_only[-1] - truth[-1]):.3f} rad（零偏越积越大）")
print(f"  只用加速度计：均方根误差 {rms(acc):.3f} rad（每一帧都抖）")
print(f"  互补滤波 α = 0.95：均方根误差 {rms(fused):.3f} rad")

In [ ]:
print("\n公式表：针孔投影（编的：f = 600 像素，主点 (320, 240)）")
u, v = project(np.array([0.1, -0.05, 1.0]), 600, 600, 320, 240)
print(f"  相机系里 (0.1, −0.05, 1.0) m 的点 → 像素 (u, v) = ({u:.0f}, {v:.0f})")

## 3. 代码里才看得到的

- 陀螺的零偏 0.02 rad/s 看起来很小，积 10 秒就是 0.2 rad（约 11°）。只积分的误差是一直往上涨的。
- 编数据时用了 `np.random.default_rng(0)` 固定随机种子，所以每次跑出来的噪声一样，测试才能核对。

## 4. 自己改一改

- 把 `fake_imu` 里的零偏 0.02 改成 0，看"只积分陀螺"还漂不漂。
- 把传送带速度改成 1 m/s、延迟改成 150 ms，算一下差多少，对照动画里"几乎每次夹空"。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 向内看编码器、IMU、力；向外看相机、雷达、触觉。
> 传感器不怕不准，怕不知道在哪、不知道是什么时候：先标定，再对时。